# BCO7006 — Session 9
# NumPy Foundations
**Duration:** 35 min lecture

You've been using NumPy without knowing it — pandas is built on top of NumPy arrays.

By the end of this notebook you will be able to:
1. Create arrays — `np.array`, `arange`, `linspace`, `zeros/ones`
2. Understand `shape`, `ndim`, `dtype`, `reshape`
3. Index and slice (1D and 2D)
4. Use **boolean masks** to filter
5. Understand **broadcasting** rules
6. Appreciate **why NumPy is fast** (vectorisation)

## 1. Setup and motivation

In [1]:
import numpy as np
import time
print("NumPy version:", np.__version__)

NumPy version: 2.4.4


### Why NumPy? A speed demo.

We'll square every number from 0 to 9,999,999. Once with a Python list, once with a NumPy array.

In [2]:
N = 10_000_000

# Python list approach
start = time.time()
result_py = [x*x for x in range(N)]
py_time = time.time() - start

# NumPy approach
start = time.time()
arr = np.arange(N)
result_np = arr * arr
np_time = time.time() - start

print(f"Python list: {py_time:.3f} s")
print(f"NumPy array: {np_time:.3f} s")
print(f"NumPy is ~{py_time/np_time:.0f}x faster")

Python list: 1.917 s
NumPy array: 0.653 s
NumPy is ~3x faster


**Why so fast?**
1. NumPy arrays are stored in contiguous memory (cache-friendly)
2. Operations happen in compiled C code, not interpreted Python
3. A single operation works on the whole array at once — no Python loop

This is called **vectorisation**. It's the whole point of NumPy.

## 2. Creating arrays

In [3]:
# From a Python list
a = np.array([1, 2, 3, 4, 5])
print(a)
print("Type:", type(a).__name__)
print("dtype:", a.dtype)

[1 2 3 4 5]
Type: ndarray
dtype: int64


In [4]:
# arange — like range() but returns an array
b = np.arange(0, 10, 2)  # start, stop (exclusive), step
print(b)

[0 2 4 6 8]


In [5]:
# linspace — N evenly-spaced numbers from start to stop (INCLUSIVE)
c = np.linspace(0, 1, 5)  # 5 points between 0 and 1
print(c)

[0.   0.25 0.5  0.75 1.  ]


In [6]:
# zeros, ones, full — fill with a value
print(np.zeros(5))
print(np.ones((2, 3)))         # 2 rows, 3 columns
print(np.full((2, 2), 7.5))

[0. 0. 0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[7.5 7.5]
 [7.5 7.5]]


In [7]:
# Random — useful for simulations
np.random.seed(0)  # makes results reproducible
print("Uniform [0, 1):", np.random.rand(4))
print("Standard normal:", np.random.randn(4))
print("Random ints 0-9:", np.random.randint(0, 10, size=4))

Uniform [0, 1): [0.5488135  0.71518937 0.60276338 0.54488318]
Standard normal: [ 1.86755799 -0.97727788  0.95008842 -0.15135721]
Random ints 0-9: [8 1 6 7]


## 3. Shape, dimensions, dtype

In [8]:
arr2d = np.array([[1, 2, 3],
                  [4, 5, 6]])
print("Array:")
print(arr2d)
print("\nshape:", arr2d.shape)   # (rows, cols)
print("ndim:", arr2d.ndim)       # number of dimensions
print("size:", arr2d.size)       # total elements
print("dtype:", arr2d.dtype)

Array:
[[1 2 3]
 [4 5 6]]

shape: (2, 3)
ndim: 2
size: 6
dtype: int64


In [9]:
# Reshape — same data, different shape
flat = np.arange(12)
print("Flat:", flat)

grid = flat.reshape(3, 4)
print("\n3x4 grid:")
print(grid)

# -1 means "figure it out"
auto = flat.reshape(2, -1)
print("\n2 x auto:")
print(auto)
print("Shape:", auto.shape)

Flat: [ 0  1  2  3  4  5  6  7  8  9 10 11]

3x4 grid:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

2 x auto:
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
Shape: (2, 6)


## 4. Indexing and slicing

In [10]:
# 1D — just like Python lists
a = np.arange(10) * 10
print(a)
print("a[0]:", a[0])
print("a[-1]:", a[-1])
print("a[2:6]:", a[2:6])
print("a[::2]:", a[::2])  # every 2nd

[ 0 10 20 30 40 50 60 70 80 90]
a[0]: 0
a[-1]: 90
a[2:6]: [20 30 40 50]
a[::2]: [ 0 20 40 60 80]


In [11]:
# 2D — comma-separated indices: arr[row, col]
m = np.arange(20).reshape(4, 5)
print("Matrix:")
print(m)
print("\nm[1, 2]:", m[1, 2])
print("\nFirst row:", m[0])
print("Last column:", m[:, -1])
print("\nTop-left 2x3 block:")
print(m[0:2, 0:3])

Matrix:
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

m[1, 2]: 7

First row: [0 1 2 3 4]
Last column: [ 4  9 14 19]

Top-left 2x3 block:
[[0 1 2]
 [5 6 7]]


## 5. Boolean masks — the heart of NumPy filtering

In [12]:
prices = np.array([12.5, 7.0, 19.99, 5.5, 31.0, 8.75, 14.0])

# Comparison returns a boolean array
mask = prices > 10
print("Prices:", prices)
print("Mask:  ", mask)

# Use the mask to filter
print("\nPrices > 10:", prices[mask])

Prices: [12.5   7.   19.99  5.5  31.    8.75 14.  ]
Mask:   [ True False  True False  True False  True]

Prices > 10: [12.5  19.99 31.   14.  ]


In [13]:
# Inline form — same thing, one line
print(prices[prices > 10])

[12.5  19.99 31.   14.  ]


In [14]:
# Multiple conditions — use &, |, ~ with PARENTHESES (same rule as pandas)
selected = prices[(prices > 10) & (prices < 20)]
print("Between 10 and 20:", selected)

Between 10 and 20: [12.5  19.99 14.  ]


In [15]:
# np.where — vectorised if/else
discounted = np.where(prices > 15, prices * 0.9, prices)
print("Original:    ", prices)
print("After 10% off items >15:")
print(discounted)

Original:     [12.5   7.   19.99  5.5  31.    8.75 14.  ]
After 10% off items >15:
[12.5    7.    17.991  5.5   27.9    8.75  14.   ]


## 6. Broadcasting — the rule that makes NumPy elegant

In [16]:
# Scalar broadcasts to every element
a = np.array([1, 2, 3, 4])
print(a + 10)
print(a * 2)

[11 12 13 14]
[2 4 6 8]


In [17]:
# 1D array + 1D array — element-wise
prices = np.array([10, 20, 30, 40])
discounts = np.array([1, 2, 3, 4])
print(prices - discounts)

[ 9 18 27 36]


In [18]:
# Where broadcasting REALLY shines: different shapes
# A 3x4 matrix + a length-4 row vector — the row gets added to every row
matrix = np.arange(12).reshape(3, 4)
row = np.array([10, 100, 1000, 10000])

print("Matrix:")
print(matrix)
print("\nRow:", row)
print("\nMatrix + row:")
print(matrix + row)

Matrix:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Row: [   10   100  1000 10000]

Matrix + row:
[[   10   101  1002 10003]
 [   14   105  1006 10007]
 [   18   109  1010 10011]]


### Broadcasting rules (simplified)

When operating on two arrays:
1. Align shapes from the **right**
2. Two dimensions match if they're **equal** OR **one of them is 1**
3. Missing dimensions on the left are treated as 1

```
Matrix:  3 × 4
Row:         4    →  treated as (1 × 4) → stretched to (3 × 4). 
```

If shapes can't be aligned, you get an error — which is good, it usually means a real bug.

In [19]:
# Common pitfall: shapes don't broadcast
try:
    np.arange(6).reshape(2, 3) + np.arange(4)
except ValueError as e:
    print("Error:", e)

Error: operands could not be broadcast together with shapes (2,3) (4,) 


## 7. Aggregations along axes

In [20]:
m = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])
print("Matrix:")
print(m)
print("\nSum of everything:", m.sum())
print("Sum down each column (axis=0):", m.sum(axis=0))
print("Sum across each row  (axis=1):", m.sum(axis=1))

Matrix:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

Sum of everything: 78
Sum down each column (axis=0): [15 18 21 24]
Sum across each row  (axis=1): [10 26 42]


**Mnemonic:** `axis=0` collapses rows (operates DOWN columns). `axis=1` collapses columns (operates ACROSS rows). The axis you name is the one that DISAPPEARS.

In [21]:
# Same pattern works for mean, std, min, max, argmin, argmax, etc.
print("Mean of each column:", m.mean(axis=0))
print("Max of each row:    ", m.max(axis=1))
print("Index of overall max:", m.argmax())

Mean of each column: [5. 6. 7. 8.]
Max of each row:     [ 4  8 12]
Index of overall max: 11


## 8. Handling NaN

In [22]:
data = np.array([1.0, 2.0, np.nan, 4.0, 5.0])
print("With NaN:")
print("  np.mean:", np.mean(data))      # contaminated
print("  np.nanmean:", np.nanmean(data))  # ignores NaN
print("  Is NaN?:", np.isnan(data))

With NaN:
  np.mean: nan
  np.nanmean: 3.0
  Is NaN?: [False False  True False False]


## 9. Summary

| Task | Method |
|---|---|
| Create from list | `np.array([...])` |
| Create range | `np.arange(start, stop, step)` |
| Evenly spaced | `np.linspace(start, stop, n)` |
| Fill | `np.zeros`, `np.ones`, `np.full` |
| Random | `np.random.rand`, `randn`, `randint` |
| Shape info | `.shape`, `.ndim`, `.size`, `.dtype` |
| Reshape | `arr.reshape(rows, cols)` or `(-1, n)` |
| 2D index | `arr[row, col]` |
| Filter | `arr[mask]` where `mask = arr > 5` |
| Vectorised if/else | `np.where(cond, a, b)` |
| Aggregate column-wise | `arr.sum(axis=0)` |
| Aggregate row-wise | `arr.sum(axis=1)` |
| NaN-safe stats | `np.nanmean`, `np.nansum`, `np.nanstd` |

**Next:** the exploratory programming activity goes deeper, and the pair programming activity makes you feel the speed of vectorisation yourself.